# What is a recorded field worth?

A worked example of [`fieldvalue`](../fieldvalue/README.md) on **UCI Adult**
(`doi:10.24432/C5XW20`) -- a public dataset that is *not* one of the event logs in
the accompanying paper.

The question: **what is `occupation` worth** for predicting whether income exceeds
$50k?  `occupation` is the expensive field here.  It has to be asked about and coded.
`age`, `sex`, `race` and `native_country` come free with the sampling frame, and
`education` comes with the enrolment record.

The point of this notebook is that *the question as asked has no answer*.  It has a
surface.


In [ ]:
import sys, numpy as np, pandas as pd
sys.path.insert(0, '..')
from examples.worked_example import load, CHEAP, FREE, EXPENSIVE
from fieldvalue import surface, SingleNumberRefused

d = load()
X = d[CHEAP + ['hours_band', EXPENSIVE]]
y = d._y.values
len(d), y.mean()


## One baseline, one metric, one number

The way it is normally reported.


In [ ]:
s0 = surface(X, y, feature=EXPENSIVE,
             baselines={'free': FREE, 'free+education': CHEAP},
             metrics=['auc'], thresholds=(), population=(1.0,))
row = s0.at(metric='auc', population=1.0)
print(f"V(occupation | free)           = {row.V_lo:+.4f} AUC")
print(f"V(occupation | free+education) = {row.V_hi:+.4f} AUC")
print(f"the reduction R                = {row.R:+.3f}")


## Now vary the four choices

Four baselines, four metrics, sixteen operating points, four population levels.


In [ ]:
s = surface(X, y, feature=EXPENSIVE,
            baselines={'nothing': [], 'free': FREE,
                       'free+education': CHEAP,
                       'free+education+hours': CHEAP + ['hours_band']},
            metrics=['auc', 'ap', 'brier_skill', 'nagelkerke'],
            thresholds=tuple(np.round(np.arange(0.05, 0.8001, 0.05), 4)),
            population=(1.00, 0.75, 0.50, 0.25), seed=20260819)
s.report()


## How much does each choice move the answer?


In [ ]:
s.spread()


## The signature figure

Baseline on one axis, metric on the other; each cell is the reduction.


In [ ]:
s.plot();


## And the refusal

`float(s)` does not return a number.  That is the whole argument.


In [ ]:
try:
    float(s)
except SingleNumberRefused as e:
    print(e)


## What to take away

Every cell above is a defensible answer to *what is `occupation` worth*.  They
differ by more than the effect any one of them reports.  If a paper, a business
case or a tool evaluation gives you one number, ask which cell it came from --
and if the answer is not in the paper, the number does not mean what it appears
to mean.
